## DNN-Based Chatbot with Ollama & Scikit-learn
A conversational chatbot built using a Deep Neural Network (DNN) architecture,
developed in Python as part of my M.Sc. Data Science program (2025).

In [2]:
!pip install ollama

In [3]:
import ollama

In [4]:
dataset = []

In [5]:
import codecs

In [6]:
with codecs.open('E:/Downloads/cat-facts.txt', 'r', encoding='utf-8',errors='ignore') as fdata:
    dataset = fdata.readlines()
    print(f'Loaded {len(dataset)} entries')

Loaded 319 entries


In [7]:
EMBEDDING_MODEL = "nomic-embed-text"

LANGUAGE_MODEL = "llama3"


# Each element in the VECTOR_DB will be a tuple (chunk,embedding)
# The embedding is a list of floats, for example: [0.1, 0.04, -0.34,0.21, ...]

VECTOR_DB = []

def add_chunk_to_database(chunk):
    embedding = ollama.embed(model="nomic-embed-text", input=chunk)['embeddings'][0]
    VECTOR_DB.append((chunk, embedding))


In [8]:
    
for i, chunk in enumerate(dataset):
    add_chunk_to_database(chunk)
    print(f'Added chunk {i+1}/{len(dataset)} to the database')

Added chunk 1/319 to the database
Added chunk 2/319 to the database
Added chunk 3/319 to the database
Added chunk 4/319 to the database
Added chunk 5/319 to the database
Added chunk 6/319 to the database
Added chunk 7/319 to the database
Added chunk 8/319 to the database
Added chunk 9/319 to the database
Added chunk 10/319 to the database
Added chunk 11/319 to the database
Added chunk 12/319 to the database
Added chunk 13/319 to the database
Added chunk 14/319 to the database
Added chunk 15/319 to the database
Added chunk 16/319 to the database
Added chunk 17/319 to the database
Added chunk 18/319 to the database
Added chunk 19/319 to the database
Added chunk 20/319 to the database
Added chunk 21/319 to the database
Added chunk 22/319 to the database
Added chunk 23/319 to the database
Added chunk 24/319 to the database
Added chunk 25/319 to the database
Added chunk 26/319 to the database
Added chunk 27/319 to the database
Added chunk 28/319 to the database
Added chunk 29/319 to the dat

Added chunk 233/319 to the database
Added chunk 234/319 to the database
Added chunk 235/319 to the database
Added chunk 236/319 to the database
Added chunk 237/319 to the database
Added chunk 238/319 to the database
Added chunk 239/319 to the database
Added chunk 240/319 to the database
Added chunk 241/319 to the database
Added chunk 242/319 to the database
Added chunk 243/319 to the database
Added chunk 244/319 to the database
Added chunk 245/319 to the database
Added chunk 246/319 to the database
Added chunk 247/319 to the database
Added chunk 248/319 to the database
Added chunk 249/319 to the database
Added chunk 250/319 to the database
Added chunk 251/319 to the database
Added chunk 252/319 to the database
Added chunk 253/319 to the database
Added chunk 254/319 to the database
Added chunk 255/319 to the database
Added chunk 256/319 to the database
Added chunk 257/319 to the database
Added chunk 258/319 to the database
Added chunk 259/319 to the database
Added chunk 260/319 to the d

In [9]:
def cosine_similarity(a, b):
    dot_product = sum([x * y for x, y in zip(a, b)])
    norm_a = sum([x ** 2 for x in a]) ** 0.5
    norm_b = sum([x ** 2 for x in b]) ** 0.5
    return dot_product / (norm_a * norm_b)

In [10]:
def retrieve(query, top_n=3):
    query_embedding = ollama.embed(model=EMBEDDING_MODEL, input=query)['embeddings'][0]
    


In [11]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def retrieve(query, top_n=3):
    # Step 1: Embed the query
    query_embedding = ollama.embed(model=EMBEDDING_MODEL, input=query)['embeddings'][0]
    query_embedding = np.array(query_embedding).reshape(1, -1)  # Reshape for sklearn

    # Step 2: Compute similarities
    similarities = []
    for chunk, embedding in VECTOR_DB:
        embedding = np.array(embedding).reshape(1, -1)
        similarity = cosine_similarity(query_embedding, embedding)[0][0]
        similarities.append((chunk, similarity))

    # Step 3: Sort and return top N
    similarities.sort(key=lambda x: x[1], reverse=True)
    
    # finally, return the top N most relevant chunks
    return similarities[:top_n]


In [12]:
# Chatbot

input_query = input('Ask me a question: ')
retrieved_knowledge = retrieve(input_query)

print('Retrieved knowledge:')
for chunk, similarity in retrieved_knowledge:
    print(f' - (similarity: {similarity:.2f}) {chunk}')
    
    
instruction_prompt = f'''You are a helpful chatbot.
Use only the following pieces of context to answer the question.
Don't make up any new information:{' '.join([f' - {chunk}' for chunk, similarity in retrieved_knowledge])}'''
# print(instruction_prompt)

stream = ollama.chat(model=LANGUAGE_MODEL, messages=[{'role': 'system', 'content': instruction_prompt},{'role': 'user', 'content': input_query},],stream=True,)

# print the response from the chatbot in real-time
print('Chatbot response:')
for chunk in stream:
    print(chunk['message']['content'], end='', flush=True)

Ask me a question:  Tell me about cats?


Retrieved knowledge:
 - (similarity: 0.70) All cats are members of the family Felidae.

 - (similarity: 0.70) A cat lover is called an Ailurophilia (Greek: cat+lover).

 - (similarity: 0.70) The more cats are spoken to, the more they will speak to you.

Chatbot response:
Cats! According to our context, all cats are members of the family Felidae. That's a great starting point! Did you know that cat lovers are actually called Ailurophiles (Greek: cat+lover)? It's a fascinating field of study! And did you know that the more you talk to cats, the more they'll respond?